# Programming Assignment: Outlier Investigation

Welcome to the **Outlier Investigation** assignment!

Outliers — data points that deviate significantly from the majority — can distort model training, inflate error metrics, and mask true patterns. Knowing *when* and *how* to detect them is a core data-cleaning skill.

This notebook covers three complementary detection strategies:

| Method | When to Use |
|--------|-------------|
| IQR | Skewed distributions, robust to extremes |
| Z-score | Approximately normal distributions |
| Isolation Forest | High-dimensional data, no distribution assumptions |

**What You Will Do in This Assignment:**

* Implement **IQR-based** outlier detection using quartile boundaries
* Implement **Z-score-based** outlier detection using standard deviation
* Implement **Isolation Forest** outlier detection for multivariate data
* Remove detected outliers and return a clean DataFrame
* Create a comprehensive **outlier summary report** across all numeric columns

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

---

## Table of Contents
- [Imports](#imports)
- [1 - Understanding the Data](#1---understanding-the-data)
- [2 - IQR Outlier Detection](#2---iqr-outlier-detection)
    - **[Exercise 1 - `detect_iqr_outliers`](#exercise-1---detect_iqr_outliers)**
- [3 - Z-Score Outlier Detection](#3---z-score-outlier-detection)
    - **[Exercise 2 - `detect_zscore_outliers`](#exercise-2---detect_zscore_outliers)**
- [4 - Isolation Forest Outlier Detection](#4---isolation-forest-outlier-detection)
    - **[Exercise 3 - `detect_isolation_forest_outliers`](#exercise-3---detect_isolation_forest_outliers)**
- [5 - Removing Outliers](#5---removing-outliers)
    - **[Exercise 4 - `remove_outliers`](#exercise-4---remove_outliers)**
- [6 - Outlier Summary Report](#6---outlier-summary-report)
    - **[Exercise 5 - `create_outlier_summary`](#exercise-5---create_outlier_summary)**
- [7 - Visualizing Results](#7---visualizing-results)

<a name='imports'></a>
## Imports

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import IsolationForest

In [ ]:
import helper_utils
import unittests

<a name='1---understanding-the-data'></a>
## 1 - Understanding the Data

We generate a synthetic sales dataset with injected outliers so that you can verify your detection functions against a ground truth. The dataset contains numeric columns representing sales metrics; a fraction of rows have been deliberately shifted to extreme values.

Run the cells below to load the data and inspect its distribution before writing any detection logic.

In [ ]:
df = helper_utils.generate_sales_dataset(n_samples=300, outlier_rate=0.05, random_state=42)
print(f"Dataset shape: {df.shape}")
print(f"\nBasic statistics:")
df.describe().round(2)

In [ ]:
helper_utils.visualize_outliers_boxplot(df)

<a name='2---iqr-outlier-detection'></a>
## 2 - IQR Outlier Detection

The **Interquartile Range (IQR)** method flags a value as an outlier when it falls more than `factor × IQR` below Q1 or above Q3. With the default `factor=1.5`, the bounds are:

$$\text{lower} = Q_1 - 1.5 \times \text{IQR}, \qquad \text{upper} = Q_3 + 1.5 \times \text{IQR}$$

This method is robust because the quartiles themselves are not influenced by extreme values.

<a name='exercise-1---detect_iqr_outliers'></a>
### **Exercise 1 - `detect_iqr_outliers`**

**Your Task:**
Implement a function that detects outliers in a single DataFrame column using the IQR method.

**Requirements:**
- Return a **boolean `pd.Series`** with the same index as `df`, where `True` marks an outlier.
- Compute Q1 and Q3 using the `.quantile()` method at `0.25` and `0.75`.
- Calculate IQR as `Q3 - Q1`.
- Flag any value below `Q1 - factor * IQR` or above `Q3 + factor * IQR`.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

**Step-by-step guidance:**

1. Extract the target column: `series = df[column]`.
2. Compute quartiles:
   ```python
   Q1 = series.quantile(0.25)
   Q3 = series.quantile(0.75)
   ```
3. Calculate IQR: `IQR = Q3 - Q1`.
4. Define bounds:
   ```python
   lower = Q1 - factor * IQR
   upper = Q3 + factor * IQR
   ```
5. Return a boolean mask: `(series < lower) | (series > upper)`.

</details>

In [ ]:
# GRADED FUNCTION: detect_iqr_outliers
def detect_iqr_outliers(df, column, factor=1.5):
    """
    Detect outliers using the Interquartile Range (IQR) method.

    Args:
        df (pd.DataFrame): Input DataFrame.
        column (str): Column name to check for outliers.
        factor (float): IQR multiplier for bounds (default 1.5).

    Returns:
        pd.Series: Boolean Series where True indicates an outlier.
    """
    ### START CODE HERE ###

    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")

    ### END CODE HERE ###

In [ ]:
iqr_mask = detect_iqr_outliers(df, 'sales_amount')
print(f"IQR outliers in sales_amount: {iqr_mask.sum()}")
print(f"Outlier percentage: {iqr_mask.mean()*100:.1f}%")

In [ ]:
unittests.exercise_1(detect_iqr_outliers)

<a name='3---z-score-outlier-detection'></a>
## 3 - Z-Score Outlier Detection

The **Z-score** method measures how many standard deviations a value lies from the column mean:

$$z = \frac{x - \mu}{\sigma}$$

A value is flagged when $|z| > \text{threshold}$ (commonly 3.0). This method works best when the column is approximately normally distributed.

<a name='exercise-2---detect_zscore_outliers'></a>
### **Exercise 2 - `detect_zscore_outliers`**

**Your Task:**
Implement outlier detection based on the absolute Z-score of a column.

**Requirements:**
- Return a **boolean `pd.Series`** aligned to `df.index`.
- Handle `NaN` values: drop them before computing Z-scores, but keep their rows as `False` in the output.
- Use `scipy.stats.zscore` for the score computation.
- Flag values whose absolute Z-score exceeds `threshold`.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

**Step-by-step guidance:**

1. Drop NaNs: `col_data = df[column].dropna()`.
2. Compute Z-scores: `z_scores = np.abs(stats.zscore(col_data))`.
3. Initialise a `False` mask over the full DataFrame index:
   ```python
   mask = pd.Series(False, index=df.index)
   ```
4. Assign results only to the non-NaN positions:
   ```python
   mask[col_data.index] = z_scores > threshold
   ```
5. Return `mask`.

</details>

In [ ]:
# GRADED FUNCTION: detect_zscore_outliers
def detect_zscore_outliers(df, column, threshold=3.0):
    """
    Detect outliers using Z-score method.

    Args:
        df (pd.DataFrame): Input DataFrame.
        column (str): Column to analyze.
        threshold (float): Z-score threshold (default 3.0).

    Returns:
        pd.Series: Boolean Series where True indicates an outlier.
    """
    ### START CODE HERE ###

    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")

    ### END CODE HERE ###

In [ ]:
zscore_mask = detect_zscore_outliers(df, 'sales_amount')
print(f"Z-score outliers in sales_amount: {zscore_mask.sum()}")

In [ ]:
unittests.exercise_2(detect_zscore_outliers)

<a name='4---isolation-forest-outlier-detection'></a>
## 4 - Isolation Forest Outlier Detection

**Isolation Forest** is a tree-based anomaly detection algorithm that works by randomly partitioning the feature space. Anomalies are isolated faster (fewer splits needed) than normal points, so they receive a lower anomaly score.

Unlike IQR or Z-score, Isolation Forest operates in **multivariate space**: it considers all selected columns simultaneously, making it ideal for detecting outliers that are unusual in combination even if each individual value looks normal.

<a name='exercise-3---detect_isolation_forest_outliers'></a>
### **Exercise 3 - `detect_isolation_forest_outliers`**

**Your Task:**
Implement multivariate outlier detection using `sklearn`'s `IsolationForest`.

**Requirements:**
- If `columns` is `None`, default to all numeric columns.
- Drop rows with NaN values in the selected columns before fitting.
- Return a **boolean `pd.Series`** aligned to `df.index`; rows excluded due to NaN should be `False`.
- `IsolationForest.fit_predict` returns `-1` for outliers and `1` for inliers.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

**Step-by-step guidance:**

1. Select numeric columns if `columns is None`:
   ```python
   columns = df.select_dtypes(include='number').columns.tolist()
   ```
2. Extract and drop NaN rows: `X = df[columns].dropna()`.
3. Fit and predict:
   ```python
   iso = IsolationForest(contamination=contamination, random_state=random_state)
   preds = iso.fit_predict(X)
   ```
4. Build a `False` Series over `df.index`, then set `preds == -1` for rows in `X.index`.

</details>

In [ ]:
# GRADED FUNCTION: detect_isolation_forest_outliers
def detect_isolation_forest_outliers(df, columns=None, contamination=0.1, random_state=42):
    """
    Detect outliers using Isolation Forest algorithm.

    Args:
        df (pd.DataFrame): Input DataFrame.
        columns (list): Columns to use (default: all numeric).
        contamination (float): Expected proportion of outliers.
        random_state (int): Random seed.

    Returns:
        pd.Series: Boolean Series where True indicates an outlier.
    """
    ### START CODE HERE ###

    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")

    ### END CODE HERE ###

In [ ]:
iso_mask = detect_isolation_forest_outliers(df)
print(f"Isolation Forest outliers: {iso_mask.sum()}")

In [ ]:
unittests.exercise_3(detect_isolation_forest_outliers)

<a name='5---removing-outliers'></a>
## 5 - Removing Outliers

Once you have identified which rows are outliers you need to remove them cleanly and return a DataFrame that is ready for downstream modelling. The key requirements are:
- Keep only rows where the mask is `False`.
- Reset the integer index so that the cleaned DataFrame starts at 0.

<a name='exercise-4---remove_outliers'></a>
### **Exercise 4 - `remove_outliers`**

**Your Task:**
Implement a function that removes outlier rows from a DataFrame using a boolean mask.

**Requirements:**
- Keep rows where `outlier_mask` is `False`.
- Reset the index of the returned DataFrame (`drop=True`).
- Do **not** modify the original DataFrame.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

**Step-by-step guidance:**

1. Invert the mask with `~outlier_mask` to select non-outlier rows.
2. Filter: `df[~outlier_mask]`.
3. Reset index: `.reset_index(drop=True)`.
4. Return the result in a single expression.

</details>

In [ ]:
# GRADED FUNCTION: remove_outliers
def remove_outliers(df, outlier_mask):
    """
    Remove rows identified as outliers.

    Args:
        df (pd.DataFrame): Input DataFrame.
        outlier_mask (pd.Series): Boolean mask where True means outlier.

    Returns:
        pd.DataFrame: DataFrame with outlier rows removed and index reset.
    """
    ### START CODE HERE ###

    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")

    ### END CODE HERE ###

In [ ]:
iqr_mask_demo = detect_iqr_outliers(df, 'sales_amount')
df_no_outliers = remove_outliers(df, iqr_mask_demo)
print(f"Original shape : {df.shape}")
print(f"Cleaned shape  : {df_no_outliers.shape}")
print(f"Rows removed   : {df.shape[0] - df_no_outliers.shape[0]}")

In [ ]:
unittests.exercise_4(remove_outliers)

<a name='6---outlier-summary-report'></a>
## 6 - Outlier Summary Report

A summary report gives stakeholders a quick, interpretable view of data quality. Here you will build a function that runs IQR detection across every numeric column, unions the outlier flags, and reports aggregate statistics.

<a name='exercise-5---create_outlier_summary'></a>
### **Exercise 5 - `create_outlier_summary`**

**Your Task:**
Create a summary dictionary of outlier statistics across all numeric columns using the IQR method.

**Requirements:**
- If `columns` is `None`, use all numeric columns.
- A row is counted as an outlier if it is flagged in **any** column (union of per-column masks).
- Return a `dict` with exactly these keys:
  - `'total_outliers'` — number of outlier rows (int)
  - `'clean_rows'` — number of non-outlier rows (int)
  - `'outlier_pct'` — percentage of outlier rows rounded to 2 decimal places (float)

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

**Step-by-step guidance:**

1. Select numeric columns if needed.
2. Initialise a `False` Series: `all_outliers = pd.Series(False, index=df.index)`.
3. For each column, compute the IQR mask and combine with `|`:
   ```python
   all_outliers = all_outliers | mask
   ```
4. Count: `total = int(all_outliers.sum())`.
5. Build and return the dict:
   ```python
   return {
       'total_outliers': total,
       'clean_rows': len(df) - total,
       'outlier_pct': round(total / len(df) * 100, 2),
   }
   ```

</details>

In [ ]:
# GRADED FUNCTION: create_outlier_summary
def create_outlier_summary(df, columns=None):
    """
    Create a summary report of outliers across all numeric columns.

    Args:
        df (pd.DataFrame): Input DataFrame.
        columns (list): Columns to check (default: all numeric).

    Returns:
        dict: {'total_outliers': int, 'clean_rows': int, 'outlier_pct': float}
    """
    ### START CODE HERE ###

    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")

    ### END CODE HERE ###

In [ ]:
summary = create_outlier_summary(df)
print("Outlier Summary Report")
print("=======================")
for key, value in summary.items():
    print(f"  {key}: {value}")

In [ ]:
unittests.exercise_5(create_outlier_summary)

## 7 - Visualizing Results

Let's use the functions you've implemented to clean the dataset and visually compare the distribution before and after outlier removal.

In [ ]:
iqr_mask_final = detect_iqr_outliers(df, 'sales_amount')
df_cleaned = remove_outliers(df, iqr_mask_final)
helper_utils.plot_outlier_comparison(df, df_cleaned, 'sales_amount')

## Congratulations!

You have successfully completed the **Outlier Investigation** assignment. Here is a summary of what you implemented:

| Exercise | Function | Method |
|----------|----------|--------|
| 1 | `detect_iqr_outliers` | IQR fence rule |
| 2 | `detect_zscore_outliers` | Absolute Z-score |
| 3 | `detect_isolation_forest_outliers` | Isolation Forest |
| 4 | `remove_outliers` | Boolean mask filtering |
| 5 | `create_outlier_summary` | Aggregate report |

You now have a robust toolkit for outlier detection that covers univariate statistical methods as well as multivariate machine-learning-based approaches. These skills will transfer directly to real-world data cleaning pipelines!